# Profiler Exports for Batch Workflows
---

In the previous notebook we used TensorBoard to visualize PyTorch Profiler output interactively.  That works well in a JupyterLab environment with port forwarding, but most production training happens on **headless Slurm nodes** where setting up live TensorBoard access is friction-heavy.

Fortunately, PyTorch Profiler can export profiling data to several portable formats that you can download via `scp` and analyze locally.  This notebook covers four export options:

1. **Text tables** — human-readable summaries for logs and terminal output
2. **Chrome/Perfetto traces** — interactive timelines viewable in a local browser
3. **Memory timelines** — self-contained HTML for diagnosing OOM issues
4. **Flame graph stacks** — hierarchical views of where time is spent

By the end of this notebook you will know how to generate all four and when to use each.

## When to Use File-Based Exports

| Scenario | Best Export |
|----------|-------------|
| Quick check in Slurm logs | Text table |
| Deep-dive timeline analysis | Chrome trace → Perfetto |
| Debugging OOM or memory fragmentation | Memory timeline HTML |
| Understanding call-stack hotspots | Flame graph stacks |

All of these can be generated from the same profiler run — you just call different export methods after the profiling context closes.

## Setup: Create an Output Directory

We will write all export files to a single directory.  In a Slurm job you would typically write to `$SLURM_SUBMIT_DIR` or a scratch directory.

In [ ]:
from pathlib import Path

export_dir = Path("/workspace/reports/profiler_exports")
export_dir.mkdir(parents=True, exist_ok=True)
print(f"Export directory: {export_dir}")

## 1. Human-Readable Text Tables

The simplest export is a formatted text table showing the most expensive operations.  This is perfect for:

- Quick sanity checks in Slurm job logs
- Comparing runs without leaving the terminal
- Embedding in reports or documentation

After profiling completes, call `prof.key_averages().table()` to get a formatted string:

In [ ]:
import torch
import torchvision
import torchvision.transforms as T
import warnings

# Suppress profiler cycle warning
warnings.filterwarnings("ignore", message=".*Profiler clears events.*")

# Quick setup: model, data, optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torchvision.models.resnet18(weights=None, num_classes=10).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = torch.nn.CrossEntropyLoss()

transform = T.Compose([T.Resize(224), T.ToTensor(), T.Normalize((0.5,), (0.5,))])
dataset = torchvision.datasets.CIFAR10(root="../data", train=True, download=True, transform=transform)
loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
loader_iter = iter(loader)

In [ ]:
# Run profiler for a few steps
with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
    schedule=torch.profiler.schedule(wait=1, warmup=2, active=5, repeat=1),
    record_shapes=True,
) as prof:
    for step in range(10):
        try:
            x, y = next(loader_iter)
        except StopIteration:
            loader_iter = iter(loader)
            x, y = next(loader_iter)
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        
        prof.step()

# Export text table sorted by CUDA time
table_text = prof.key_averages().table(sort_by="cuda_time_total", row_limit=20)
print(table_text)

In [ ]:
# Save to file for Slurm logs
with open(export_dir / "profile_summary.txt", "w") as f:
    f.write("=" * 80 + "\n")
    f.write("PyTorch Profiler Summary (sorted by cuda_time_total)\n")
    f.write("=" * 80 + "\n\n")
    f.write(table_text)

print(f"Saved: {export_dir / 'profile_summary.txt'}")

**Sorting options:** You can sort by `cpu_time_total`, `cuda_time_total`, `cpu_memory_usage`, `cuda_memory_usage`, `self_cpu_time_total`, or `self_cuda_time_total`.

---
## 2. Chrome / Perfetto Traces

`export_chrome_trace()` writes the same event stream TensorBoard's Trace view uses, in a portable JSON format that you can open locally.  The most reliable use case in batch environments is Perfetto's bottom-pane **flame graph**, which works regardless of how the underlying timestamps land; the timeline view at the top is more sensitive to environment setup (see the heads-up at the end of this section).

**How to view it:**
1. Download the `.json` or `.json.gz` file to your local machine
2. Open [ui.perfetto.dev](https://ui.perfetto.dev/) in your browser — it auto-decompresses `.json.gz`.  `chrome://tracing` works too but does *not* decompress `.gz`, so unzip first if you go that route.
3. Drag and drop the file

In [ ]:
# Export Chrome trace (gzip-compressed to save space)
trace_path = export_dir / "timeline_trace.json.gz"
prof.export_chrome_trace(str(trace_path))
print(f"Saved: {trace_path}")
print(f"View at: https://ui.perfetto.dev/")

**What you will see in Perfetto:**

- **Flame graph (bottom pane)** — aggregates by call stack and self-time.  Works in every environment we've tested.  This is where you'll most reliably see the shape of where time goes.
- **Timeline (top pane)** — CPU threads and GPU streams as separate tracks, with each CUDA kernel as a colored bar.  When it renders, it's the best view for spotting CPU↔GPU sync points and concurrency issues.

**The timeline view does not always render.**  In many cloud/container hosts CUPTI's activity API is restricted at the driver level (`NVreg_RestrictProfilingToAdminUsers=1`) even with `--cap-add=SYS_ADMIN`.  PyTorch still records that each kernel ran, but the GPU events come back with `ts=0` while CPU events sit at process-clock time.  Perfetto computes a viewport spanning the gap and the timeline appears blank at every zoom level.  If you see this, fall back to the flame graph pane, or use **Nsight Systems** (next lab) for timeline analysis — it takes a different permission path.

---
## 3. Interactive Memory Timelines

If you are debugging Out-of-Memory (OOM) errors or trying to understand memory fragmentation, PyTorch can export a self-contained HTML file showing memory allocation over time.

**Requirements:**
- You must set `profile_memory=True` in the profiler configuration
- The export creates a standalone `.html` file you can open in any browser

In [ ]:
# Re-run with memory profiling enabled
loader_iter = iter(loader)

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
    schedule=torch.profiler.schedule(wait=1, warmup=2, active=5, repeat=1),
    record_shapes=True,
    profile_memory=True,  # Required for memory timeline
) as prof_mem:
    for step in range(10):
        try:
            x, y = next(loader_iter)
        except StopIteration:
            loader_iter = iter(loader)
            x, y = next(loader_iter)
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        
        prof_mem.step()

In [ ]:
# Export memory timeline HTML
memory_path = export_dir / "memory_timeline.html"
prof_mem.export_memory_timeline(str(memory_path), device="cuda:0")
print(f"Saved: {memory_path}")
print("Download and open in any browser to see interactive memory graph")

**What you will see:**

- Total GPU memory allocation over time
- Memory fragmentation visualization
- Which tensors are allocated at each point
- Interactive zoom and hover for details

This is invaluable when debugging OOM errors — you can see exactly when memory spikes and what tensors are responsible.

---
## 4. Flame Graph Stacks

Flame graphs show a hierarchical view of where execution time is spent across your model's call stack.  PyTorch exports this as a "folded stacks" text file that you can convert to an interactive SVG.

**Requirements:**
- You must set `with_stack=True` in the profiler configuration
- Use [FlameGraph](https://github.com/brendangregg/FlameGraph) tools to convert to SVG

In [ ]:
# Re-run with stack tracing enabled
loader_iter = iter(loader)

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
    schedule=torch.profiler.schedule(wait=1, warmup=2, active=5, repeat=1),
    record_shapes=True,
    with_stack=True,  # Required for flame graphs
) as prof_stack:
    for step in range(10):
        try:
            x, y = next(loader_iter)
        except StopIteration:
            loader_iter = iter(loader)
            x, y = next(loader_iter)
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        
        prof_stack.step()

In [ ]:
# Export folded stacks for flame graph generation
stacks_path = export_dir / "profiler_stacks.txt"
prof_stack.export_stacks(str(stacks_path), metric="self_cuda_time_total")
print(f"Saved: {stacks_path}")

**To generate a flame graph SVG:**

```bash
# Clone FlameGraph tools (one time)
git clone https://github.com/brendangregg/FlameGraph.git

# Generate SVG from the stacks file
./FlameGraph/flamegraph.pl profiler_stacks.txt > flamegraph.svg
```

The resulting SVG is interactive — hover over any bar to see the full call stack and time spent.

---
## Putting It All Together: A Slurm-Ready Pattern

Here is a template that generates all four exports in one profiling run.  Copy this pattern into your training scripts for batch jobs:

In [ ]:
# Complete example: all exports from one profiling run
import torch
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message=".*Profiler clears events.*")

# Output directory (use $SLURM_SUBMIT_DIR in batch jobs)
logdir = Path("/workspace/reports/slurm_profile_example")
logdir.mkdir(parents=True, exist_ok=True)

# Reset data loader
loader_iter = iter(loader)

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
    schedule=torch.profiler.schedule(wait=1, warmup=5, active=10, repeat=1),
    record_shapes=True,
    profile_memory=True,  # For memory timeline
    with_stack=True,      # For flame graphs
) as prof:
    for step in range(20):
        try:
            x, y = next(loader_iter)
        except StopIteration:
            loader_iter = iter(loader)
            x, y = next(loader_iter)
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        
        prof.step()

# --- Generate All Exports ---
print("Saving profiling artifacts...")

# 1. Human-readable summary table
with open(logdir / "profile_summary.txt", "w") as f:
    f.write(prof.key_averages().table(sort_by="cuda_time_total", row_limit=30))
print(f"  [1/4] {logdir / 'profile_summary.txt'}")

# 2. Chrome trace for Perfetto
prof.export_chrome_trace(str(logdir / "timeline_trace.json.gz"))
print(f"  [2/4] {logdir / 'timeline_trace.json.gz'}")

# 3. Memory timeline HTML
prof.export_memory_timeline(str(logdir / "memory_timeline.html"), device="cuda:0")
print(f"  [3/4] {logdir / 'memory_timeline.html'}")

# 4. Flame graph stacks
prof.export_stacks(str(logdir / "profiler_stacks.txt"), metric="self_cuda_time_total")
print(f"  [4/4] {logdir / 'profiler_stacks.txt'}")

print(f"\nDone! Download files from {logdir} to analyze locally.")

---
## Exercise: Export Your Own Profiles

Take one of your existing training scripts and add profiling exports:

1. Add `torch.profiler.profile()` with `profile_memory=True` and `with_stack=True`
2. Generate all four export types after the profiling context closes
3. Download the Chrome trace and open it in [ui.perfetto.dev](https://ui.perfetto.dev/)
4. Compare what you see in Perfetto to what TensorBoard showed

**Bonus:** Write a small shell script that runs your training with profiling and then uses FlameGraph to automatically generate an SVG.

---
## Summary: Export Methods at a Glance

| Method | Output | View With | Use Case |
|--------|--------|-----------|----------|
| `prof.key_averages().table()` | `.txt` | Terminal / text editor | Quick summary in logs |
| `prof.export_chrome_trace()` | `.json` / `.json.gz` | [ui.perfetto.dev](https://ui.perfetto.dev/) | Flame-graph hotspots (reliable); timeline view (environment-dependent) |
| `prof.export_memory_timeline()` | `.html` | Any browser | Memory debugging |
| `prof.export_stacks()` | `.txt` | FlameGraph → `.svg` | Call-stack hotspots |

All four can be generated from a single profiling run — just enable `profile_memory=True` and `with_stack=True` in your profiler configuration.

---
## Second Example: The Labeled Training Script, with All Four Exports

In the previous notebook we wrapped each section of the training step in `record_function`, but TensorBoard's Overview and Operator panes did not give us a per-section breakdown — those panes are built around the profiler's built-in categories and PyTorch operators, not user annotations.  Only the Trace view rendered the labels.

The export methods we just covered behave differently:

- `key_averages().table()` aggregates every recorded event by name, so each `record_function` block becomes its own row with its own CPU and CUDA totals.
- `export_chrome_trace()` writes the same data the Trace view shows, so the labeled spans render identically in Perfetto.
- `export_memory_timeline()` is keyed by allocator events, not annotations — labels do not appear there.
- `export_stacks()` includes `record_function` blocks as frames in the call stack, so the resulting flame graph shows `DataLoader` as a wide frame containing everything inside it.

Below is the labeled training loop from the previous notebook, this time profiling with `profile_memory=True` and `with_stack=True` and writing all four exports.  Run it and look at `profile_summary.txt` and the flame graph derived from `profiler_stacks.txt` — both will surface the DataLoader bottleneck obviously, without TensorBoard.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader as TorchDataLoader
from pathlib import Path

# Same bug-bearing config as train_v1_profile_labeled.py
BATCH_SIZE = 256
NUM_ITERS = 20
WARMUP_ITERS = 5
NUM_WORKERS = 0      # the bug
PIN_MEMORY = False   # the bug

labeled_logdir = Path("/workspace/reports/train_v1_profile_labeled_exports")
labeled_logdir.mkdir(parents=True, exist_ok=True)

torch.manual_seed(0)
labeled_device = torch.device("cuda")

transform = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.2, 0.2, 0.2),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
labeled_dataset = torchvision.datasets.CIFAR10(
    root="/workspace/data", train=True, download=False, transform=transform,
)
labeled_loader = TorchDataLoader(
    labeled_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True,
    persistent_workers=NUM_WORKERS > 0,
)

labeled_model = torchvision.models.resnet18(num_classes=10)
labeled_model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
labeled_model.maxpool = nn.Identity()
labeled_model = labeled_model.to(labeled_device)
labeled_optim = torch.optim.SGD(labeled_model.parameters(), lr=0.1, momentum=0.9)
labeled_loss_fn = nn.CrossEntropyLoss()
labeled_model.train()

labeled_iter = iter(labeled_loader)

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
    schedule=torch.profiler.schedule(wait=1, warmup=WARMUP_ITERS, active=10, repeat=1),
    record_shapes=True,
    profile_memory=True,  # for memory timeline
    with_stack=True,      # for flame graphs
) as prof_labeled:
    for step in range(NUM_ITERS):
        with torch.profiler.record_function("DataLoader"):
            x, y = next(labeled_iter)
        with torch.profiler.record_function("H2D"):
            x = x.to(labeled_device, non_blocking=True)
            y = y.to(labeled_device, non_blocking=True)
        with torch.profiler.record_function("Forward"):
            logits = labeled_model(x)
            loss = labeled_loss_fn(logits, y)
        with torch.profiler.record_function("Backward"):
            labeled_optim.zero_grad(set_to_none=True)
            loss.backward()
        with torch.profiler.record_function("Optimizer"):
            labeled_optim.step()
        prof_labeled.step()

# Sort by cpu_time_total because the bug is CPU-bound — self_cuda_time_total
# would push the DataLoader row off the top of the table.
with open(labeled_logdir / "profile_summary.txt", "w") as f:
    f.write(prof_labeled.key_averages().table(sort_by="cpu_time_total", row_limit=25))
print(f"[1/4] {labeled_logdir / 'profile_summary.txt'}")

prof_labeled.export_chrome_trace(str(labeled_logdir / "timeline_trace.json.gz"))
print(f"[2/4] {labeled_logdir / 'timeline_trace.json.gz'}")

prof_labeled.export_memory_timeline(str(labeled_logdir / "memory_timeline.html"), device="cuda:0")
print(f"[3/4] {labeled_logdir / 'memory_timeline.html'}")

# self_cpu_time_total for the flame graph for the same CPU-bound reason.
prof_labeled.export_stacks(str(labeled_logdir / "profiler_stacks.txt"), metric="self_cpu_time_total")
print(f"[4/4] {labeled_logdir / 'profiler_stacks.txt'}")

### What each export reveals about the bug

- **`profile_summary.txt`** — sorted by `cpu_time_total`, `DataLoader` is the row with by far the largest inclusive CPU time, with `Forward` / `Backward` / `Optimizer` / `H2D` appearing as separate rows beneath it carrying much smaller numbers.  The bottleneck is obvious without ever opening TensorBoard, which is exactly what you want in a Slurm log.
- **`timeline_trace.json.gz`** — drag into [ui.perfetto.dev](https://ui.perfetto.dev/) and you'll see the same labeled spans you saw in TensorBoard's Trace view: `DataLoader` taking up most of each step, with `H2D` / `Forward` / `Backward` / `Optimizer` compressed into the remainder.
- **`memory_timeline.html`** — useful for memory debugging, but won't show anything about the DataLoader bug specifically.  Allocator events are not keyed by `record_function` blocks.
- **`profiler_stacks.txt`** → flame graph SVG — `DataLoader` shows up as a wide frame containing the CIFAR-10 transforms (`RandomCrop`, `RandomHorizontalFlip`, `ColorJitter`, `ToTensor`, `Normalize`) and the PIL/torchvision call stack underneath them.  This is the export that tells you *why* DataLoader is slow, not just *that* it is.

Between `key_averages().table()` and the flame graph, batch-job profiling can surface this bug just as clearly as the interactive Trace view — sometimes more clearly, because the table gives you exact numbers and the flame graph gives you the full CPU call stack.

## <center><div style="text-align:center; color:#FF0000; border:3px solid red; height:80px;"><b><br/>[Next Notebook — AMP and the Limits of the Profiler](intro-amp.ipynb)</b></div></center>

---

## Links and Resources

- [Perfetto Trace Viewer](https://ui.perfetto.dev/) — recommended for viewing Chrome traces
- [FlameGraph tools](https://github.com/brendangregg/FlameGraph) — for generating flame graph SVGs
- [PyTorch Profiler documentation](https://pytorch.org/docs/stable/profiler.html)
- [PyTorch Profiler recipe](https://docs.pytorch.org/tutorials/recipes/recipes/profiler_recipe.html)

---

## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0).